<font size="8"> Dataset Builder For Tiny CNN </font>

---

In [ ]:
# Install dependencies
# !pip install opencv-python matplotlib tqdm numpy 

In [ ]:
# Imports
import cv2
import os
import glob
import random
import matplotlib.pyplot as plt
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from collections import defaultdict
from tqdm import tqdm

In [ ]:
# Check versions
print(f"OpenCV version: {cv2.__version__}")
print(f"Matplotlib version: {matplotlib.__version__}")
print(f"NumPy version: {np.__version__}")
print("All packages installed successfully")

---

Let's create a balanced dataset of 34×34 human face/non-face patches for training a tiny CNN that filters Haar cascade false positives. 

**~Important Note:** We use the two datasets: **PASS** for non-face images, and **LFW** for face images to train our CNN. To be able to run this note, you'll need to install them and add the missing paths in the code. You can also download your own datasets, but for that, you need to make sure to have two seperates datasets, one that includes only faces, and for the other to not include any human face or people in general.

# <font size="5"> Mining Hard Negatives from PASS Dataset: </font>

**Goal:** Run Haar cascade on 150k non-face images and save every false positive detection.

**Why:** These false positives are Haar's mistakes - exactly what our CNN needs to learn to reject.

**Output:** 34×34 patches saved to `haar-hard-negatives`

In [ ]:
# Configurations
NONFACE_SOURCE_DIR = ""  # Path to PASS (non-face) dataset
OUTPUT_DIR = "haar-hard-negatives"  # Where to save hard negatives
MAX_COUNT = 6000  # Maximum number of hard negatives

In [ ]:
# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
print(f"Configuration:")
print(f"  Source: {NONFACE_SOURCE_DIR}")
print(f"  Output: {OUTPUT_DIR}")
print(f"  Target: {MAX_COUNT} hard negatives")

In [ ]:
# Load Haar cascade
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_alt.xml"
)

In [ ]:
# Quick verification
if face_cascade.empty():
    print("ERROR: Failed to load Haar cascade")
else:
    print("Haar cascade loaded successfully")

In [ ]:
# Extract hard negatives
count = 0
for img_path in image_files:
    if count >= MAX_COUNT:
        break

    img = cv2.imread(img_path)
    if img is None:
        continue

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.1, 3, minSize=(30, 30))

    # Save ALL false positives from this image
    for x, y, w, h in faces:
        if count >= MAX_COUNT:
            break
        roi = cv2.resize(img[y : y + h, x : x + w], (34, 34))
        cv2.imwrite(f"{OUTPUT_DIR}/nonface_{count:06d}.jpg", roi)
        count += 1

        if count % 100 == 0:
            print(f"  Extracted {count}/{MAX_COUNT}")

print(f"\n Done: {count} hard negatives")

In [ ]:
# Visualising a few samples
sample_files = glob.glob(os.path.join(OUTPUT_DIR, "*.jpg"))[:12]
if sample_files:
    fig, axes = plt.subplots(3, 4, figsize=(12, 9))
    for i, ax in enumerate(axes.flat):
        if i < len(sample_files):
            img = cv2.imread(sample_files[i])
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            ax.imshow(img_rgb)
            ax.set_title(f"Hard negative {i+1}")
        ax.axis("off")
    plt.suptitle("Examples of Haar False Positives (Hard Negatives)")
    plt.tight_layout()
    plt.show()
else:
    print("No hard negatives extracted to display")

---

# <font size="5"> Bulding Final Dataset with Train/Val/Test Splits : </font>
**Goal:** Create a balanced dataset of faces and non-faces with proper train/val/test splits.
**Key Features:**
- Person-aware splitting for faces (no identity leakage)
- Hard negatives from previous step + augmentation
- All images resized to 34×34


In [ ]:
# Configurations
FACES_SOURCE = "" # Path to LFW (faces) dataset
NONFACES_SOURCE = "haar-hard-negatives"
DATASET_ROOT = "dataset"

# Split ratios
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

# Target counts
TARGET_FACES_TRAIN = 4900
TARGET_FACES_VAL = 1050
TARGET_FACES_TEST = 1050

TARGET_NONFACES_TRAIN = 4200
TARGET_NONFACES_VAL = 900
TARGET_NONFACES_TEST = 900

# Max augmentations per base non-face image
MAX_AUGS_PER_IMAGE = 3

In [ ]:
print(f"Configuration loaded:")
print(f"  Faces source: {FACES_SOURCE}")
print(f"  Non-faces source: {NONFACES_SOURCE}")
print(f"  Output root: {DATASET_ROOT}")
print(f"  Train faces target: {TARGET_FACES_TRAIN}")
print(f"  Train non-faces target: {TARGET_NONFACES_TRAIN}")

In [ ]:
# Directory creation function
def create_dirs():
    for split in ["train", "val", "test"]:
        os.makedirs(os.path.join(DATASET_ROOT, split, "faces"), exist_ok=True)
        os.makedirs(os.path.join(DATASET_ROOT, split, "nonfaces"), exist_ok=True)
    print("Directory structure created")

In [ ]:
create_dirs()

**~NOTE:** We don't have to augment faces, we have enough images, We only must not have the same person in different splits to avoid data leakage. however, augmenting is not a bad idea.

In [ ]:
# Process faces function
def process_faces():
    print("\nPROCESSING FACES (Person-aware)")
    person_folders = [
        d
        for d in os.listdir(FACES_SOURCE)
        if os.path.isdir(os.path.join(FACES_SOURCE, d))
    ]
    random.shuffle(person_folders)

    train_cut = int(len(person_folders) * TRAIN_RATIO)
    val_cut = train_cut + int(len(person_folders) * VAL_RATIO)

    splits = {
        "train": person_folders[:train_cut],
        "val": person_folders[train_cut:val_cut],
        "test": person_folders[val_cut:],
    }

    print(f"Found {len(person_folders)} unique people")
    print(f"  Train: {len(splits['train'])} people")
    print(f"  Val: {len(splits['val'])} people")
    print(f"  Test: {len(splits['test'])} people")

    split_counts = {"train": 0, "val": 0, "test": 0}

    for split, people in splits.items():
        target = {
            "train": TARGET_FACES_TRAIN,
            "val": TARGET_FACES_VAL,
            "test": TARGET_FACES_TEST,
        }[split]

        print(f"\nProcessing {split.upper()} split (target: {target} faces)")

        for person in people:
            if split_counts[split] >= target:
                break

            imgs = glob.glob(os.path.join(FACES_SOURCE, person, "*.jpg"))
            for img_path in imgs:
                if split_counts[split] >= target:
                    break

                img = cv2.imread(img_path)
                if img is None:
                    continue

                gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
                faces = face_cascade.detectMultiScale(gray, 1.1, 3, minSize=(30, 30))

                if len(faces) == 0:
                    continue

                x, y, w, h = max(faces, key=lambda f: f[2] * f[3])
                roi = cv2.resize(img[y : y + h, x : x + w], (34, 34))

                out = os.path.join(
                    DATASET_ROOT, split, "faces", f"face_{split_counts[split]:06d}.jpg"
                )
                cv2.imwrite(out, roi)
                split_counts[split] += 1

                if split_counts[split] % 500 == 0:
                    print(f"  {split}: {split_counts[split]}/{target}")

        print(f"  Final {split}: {split_counts[split]} faces")

    return split_counts

In [ ]:
# Run face processing
face_counts = process_faces()

In [ ]:
print(f"\n Face extraction complete:")
print(f"  Train: {face_counts['train']} faces")
print(f"  Val: {face_counts['val']} faces")
print(f"  Test: {face_counts['test']} faces")

<font size="4"> Proccesing non faces with augmentation: </font>

In [ ]:
# non-face augmentaion function
def get_augmentations():
    return {
        "hflip": lambda x: cv2.flip(x, 1),
        "bright_up": lambda x: cv2.convertScaleAbs(x, alpha=1.1, beta=10),
        "bright_down": lambda x: cv2.convertScaleAbs(x, alpha=0.9, beta=-10),
        "noise": lambda x: np.clip(
            x.astype(np.int16) + np.random.randint(-5, 5, x.shape, dtype=np.int16),
            0,
            255,
        ).astype(np.uint8),
    }

In [ ]:
aug_defs = get_augmentations()
print(f" Augmentation functions defined: {list(aug_defs.keys())}")

In [ ]:
# Process non-faces function
def process_nonfaces():
    print("\nPROCESSING NON-FACES")

    nonface_files = glob.glob(os.path.join(NONFACES_SOURCE, "*.jpg"))
    print(f"Found {len(nonface_files)} base non-face images")

    random.shuffle(nonface_files)

    train_cut = int(len(nonface_files) * TRAIN_RATIO)
    val_cut = train_cut + int(len(nonface_files) * VAL_RATIO)

    splits = {
        "train": nonface_files[:train_cut],
        "val": nonface_files[train_cut:val_cut],
        "test": nonface_files[val_cut:],
    }

    print(f"Base split sizes:")
    print(f"  Train: {len(splits['train'])}")
    print(f"  Val: {len(splits['val'])}")
    print(f"  Test: {len(splits['test'])}")

    targets = {
        "train": TARGET_NONFACES_TRAIN,
        "val": TARGET_NONFACES_VAL,
        "test": TARGET_NONFACES_TEST,
    }

    split_counts = {"train": 0, "val": 0, "test": 0}

    for split, base_files in splits.items():
        target = targets[split]
        print(f"\n{split.upper()} target: {target}")

        # Track how many augmentations each image has produced
        aug_count = defaultdict(int)
        used_augs = defaultdict(set)

        # 1) Copy base images first
        for img_path in base_files:
            if split_counts[split] >= target:
                break

            img = cv2.imread(img_path)
            if img is None:
                continue

            out = os.path.join(
                DATASET_ROOT,
                split,
                "nonfaces",
                f"nonface_{split_counts[split]:06d}.jpg",
            )
            cv2.imwrite(out, img)
            split_counts[split] += 1

        print(f"  Copied {split_counts[split]} base images")

        # 2) Augment until target is reached
        base_idx = 0
        base_len = len(base_files)

        while split_counts[split] < target:
            img_path = base_files[base_idx]
            fname = os.path.basename(img_path)

            # Skip if max augmentations reached
            if aug_count[fname] >= MAX_AUGS_PER_IMAGE:
                base_idx = (base_idx + 1) % base_len
                continue

            img = cv2.imread(img_path)
            if img is None:
                base_idx = (base_idx + 1) % base_len
                continue

            available = [k for k in aug_defs.keys() if k not in used_augs[fname]]

            if not available:
                base_idx = (base_idx + 1) % base_len
                continue

            aug_name = random.choice(available)
            aug_img = aug_defs[aug_name](img)

            used_augs[fname].add(aug_name)
            aug_count[fname] += 1

            out = os.path.join(
                DATASET_ROOT,
                split,
                "nonfaces",
                f"nonface_{split_counts[split]:06d}.jpg",
            )
            cv2.imwrite(out, aug_img)
            split_counts[split] += 1

            if split_counts[split] % 200 == 0:
                print(f"  {split}: {split_counts[split]}/{target}")

            base_idx = (base_idx + 1) % base_len

        print(f"  Final {split}: {split_counts[split]} non-faces")

    return split_counts

In [ ]:
# Run non-face processing
nonface_counts = process_nonfaces()

In [ ]:
print(f"\n Non-face processing complete:")
print(f"  Train: {nonface_counts['train']} non-faces")
print(f"  Val: {nonface_counts['val']} non-faces")
print(f"  Test: {nonface_counts['test']} non-faces")

<font size="5"> Dataset verification: </font> 

In [ ]:
def verify_dataset():
    print("DATASET VERIFICATION")

    total_faces = 0
    total_nonfaces = 0

    for split in ["train", "val", "test"]:
        faces_dir = os.path.join(DATASET_ROOT, split, "faces")
        nonfaces_dir = os.path.join(DATASET_ROOT, split, "nonfaces")

        f = len([f for f in os.listdir(faces_dir) if f.endswith(".jpg")])
        nf = len([f for f in os.listdir(nonfaces_dir) if f.endswith(".jpg")])

        total_faces += f
        total_nonfaces += nf

        print(f"\n{split.upper()}:")
        print(f"  Faces: {f}")
        print(f"  Non-faces: {nf}")
        print(f"  Total: {f + nf}")
        print(f"  Ratio (faces:nonfaces): 1:{nf/f:.2f}")

    print(f"TOTAL Faces: {total_faces}")
    print(f"TOTAL Non-faces: {total_nonfaces}")
    print(f"GRAND TOTAL: {total_faces + total_nonfaces}")

In [ ]:
# Run verification
verify_dataset()

It's also important to check image **dimensions**, our CNN's inputs are images of size 34x34:

In [ ]:
# Quick check of image dimensions
sample_face = cv2.imread(
    os.path.join(
        DATASET_ROOT,
        "train",
        "faces",
        os.listdir(os.path.join(DATASET_ROOT, "train", "faces"))[0],
    )
)
sample_nonface = cv2.imread(
    os.path.join(
        DATASET_ROOT,
        "train",
        "nonfaces",
        os.listdir(os.path.join(DATASET_ROOT, "train", "nonfaces"))[0],
    )
)

print(f"Face image shape: {sample_face.shape}")
print(f"Non-face image shape: {sample_nonface.shape}")

if sample_face.shape[:2] == (34, 34) and sample_nonface.shape[:2] == (34, 34):
    print("All images correctly sized to 34×34")
else:
    print("Warning: Image size mismatch!")

---

### Alhajali Jalal

---